# node2vec + UMAP embedding of the opening transposition graph

Alternative to `analysis.py`'s forceatlas2 layout: instead of a physics simulation
driven by direct edge weight (transposition count), this embeds each opening by
the *structural context* random walks see around it (node2vec-style), then
projects that embedding to 2D with UMAP.

Embedding step uses random walks -> co-occurrence counts -> truncated SVD
instead of node2vec's usual word2vec/skip-gram training: `gensim` (and the
`node2vec` PyPI package, which wraps it) has no Python 3.14 wheel and its sdist
fails to build (generated C references removed CPython internals). The
walks-to-SVD route is the well-known matrix-factorization equivalent of
DeepWalk/node2vec (NetMF), needs only numpy/scipy (already pulled in by
`umap-learn`).

Loads the cached `results/adjacency_matrix_7766.csv` (same n_games as
`images/graph_7766.png`) so communities/colors are directly comparable to that
existing render.

In [ ]:
import random
import sys
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from umap import UMAP

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))
from analysis import MIN_OCCURRENCES, build_filtered_graph, get_communities  # noqa: E402

N_GAMES = 19999

## Load cached graph data

Reuses the CSV already written by `analysis.py`'s `save_results()` -- no need to
re-run the game-parsing pipeline for this example.

In [ ]:
adjacency_matrix = pd.read_csv(
    REPO_ROOT / "results" / f"adjacency_matrix_{N_GAMES}.csv", index_col=0
)
adjacency_matrix.shape

In [ ]:
# reuse analysis.py's filtering logic directly -- same self-loop removal,
# Start-node occurrence override, and min-occurrences cutoff as plot_graph(),
# so this stays in sync with the script instead of drifting
_, undirected, occurrences = build_filtered_graph(adjacency_matrix, MIN_OCCURRENCES)

In [ ]:
print(f"{undirected.number_of_nodes()} nodes, {undirected.number_of_edges()} edges after filtering")

In [ ]:
# same call as analysis.py:plot_graph, same seed -- community IDs are directly
# comparable to the colors in images/graph_7766.png
community_of = get_communities(undirected)
len(set(community_of.values()))

## node2vec-style embedding (random walk + SVD, no gensim)

Weighted random walk at each node (probability of stepping to a neighbor
proportional to transposition count, i.e. unbiased/p=q=1 node2vec), then a
node-by-node co-occurrence matrix over a sliding window, then truncated SVD --
the same signal word2vec's skip-gram training would extract, but via linear
algebra instead of gradient descent.

In [ ]:
def random_walk(graph: nx.Graph, start: str, walk_length: int, rng: random.Random) -> list[str]:
    walk = [start]
    for _ in range(walk_length - 1):
        neighbors = list(graph.neighbors(walk[-1]))
        if not neighbors:
            break
        weights = [graph[walk[-1]][n]["weight"] for n in neighbors]
        walk.append(rng.choices(neighbors, weights=weights)[0])
    return walk


def build_embedding(
    graph: nx.Graph, dimensions: int, walk_length: int, num_walks: int, window: int, seed: int
) -> pd.DataFrame:
    rng = random.Random(seed)
    nodes = list(graph.nodes())
    if dimensions > len(nodes):
        raise ValueError(
            f"dimensions={dimensions} exceeds {len(nodes)} nodes -- SVD can't "
            "produce that many components; lower dimensions or min_occurrences"
        )
    index_of = {n: i for i, n in enumerate(nodes)}
    cooccurrence = np.zeros((len(nodes), len(nodes)))

    for node in nodes:
        for _ in range(num_walks):
            walk = random_walk(graph, node, walk_length, rng)
            for i, center in enumerate(walk):
                for j in range(max(0, i - window), min(len(walk), i + window + 1)):
                    if i != j:
                        cooccurrence[index_of[center], index_of[walk[j]]] += 1

    # log-scale dampens hub-node co-occurrence counts, same idea as the
    # weight_sqrt trick in analysis.py's plot_graph
    cooccurrence = np.log1p(cooccurrence)
    u, s, _ = np.linalg.svd(cooccurrence, full_matrices=False)
    embedding = u[:, :dimensions] * s[:dimensions]
    return pd.DataFrame(embedding, index=nodes)

In [ ]:
embeddings = build_embedding(
    undirected, dimensions=32, walk_length=20, num_walks=20, window=5, seed=0
)
embeddings.shape

## UMAP projection + interactive plot

In [ ]:
coords_2d = UMAP(n_components=2, random_state=0).fit_transform(embeddings.values)

In [ ]:
def plot_embedding(coords: np.ndarray, nodes: list[str], title: str):
    plot_df = pd.DataFrame(
        {
            "opening": nodes,
            "x": coords[:, 0],
            "y": coords[:, 1],
            "community": [str(community_of[n]) for n in nodes],
            "occurrences": [occurrences[n] for n in nodes],
        }
    )
    fig = px.scatter(
        plot_df,
        x="x",
        y="y",
        color="community",
        size="occurrences",
        hover_name="opening",
        hover_data={"occurrences": True, "community": True, "x": False, "y": False},
        title=title,
    )

    # edges as a single lines-trace, None-separated segments -- one trace for
    # all edges stays fast, at the cost of no per-edge hover/style. nodes is
    # always undirected's full (already-filtered) node set in every caller
    # below, so undirected.edges() needs no extra subgraph() filtering
    pos = dict(zip(plot_df["opening"], zip(plot_df["x"], plot_df["y"])))
    edge_x, edge_y = [], []
    for u, v in undirected.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
    fig.add_trace(
        go.Scatter(
            x=edge_x,
            y=edge_y,
            mode="lines",
            line=dict(width=0.5, color="lightgray"),
            hoverinfo="skip",
            showlegend=False,
        )
    )
    fig.data = (fig.data[-1],) + fig.data[:-1]  # edges first -> drawn underneath nodes
    fig.update_layout(width=1000, height=800)
    return fig

In [ ]:
fig = plot_embedding(
    coords_2d, list(embeddings.index), f"node2vec-style embedding + UMAP (n_games={N_GAMES})"
)
fig.show()

In [ ]:
# top-2 SVD components directly, no UMAP -- linear projection of the same
# walk embedding, cheaper but usually less visually separated than UMAP's
# nonlinear layout
coords_direct = embeddings.values[:, :2]
fig_direct = plot_embedding(
    coords_direct,
    list(embeddings.index),
    f"Direct 2D: top-2 SVD components, no UMAP (n_games={N_GAMES})",
)
fig_direct.show()

In [ ]:
# UMAP directly on the raw weighted adjacency matrix -- skips the random
# walk + co-occurrence + SVD step entirely, UMAP builds its own neighbor
# graph straight from edge weights instead of from structural/walk context
nodes_adj = list(undirected.nodes())
adjacency_raw = nx.to_numpy_array(undirected, nodelist=nodes_adj, weight="weight")
coords_raw = UMAP(n_components=2, random_state=0).fit_transform(adjacency_raw)
fig_raw = plot_embedding(
    coords_raw,
    nodes_adj,
    f"UMAP direct on adjacency, no walk/SVD preprocessing (n_games={N_GAMES})",
)
fig_raw.show()

## Compare against the existing forceatlas2 + Louvain layout

Same community IDs/colors as above -- check whether nodes that cluster by
connection strength (below) also cluster by random-walk structural similarity
(above).

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(REPO_ROOT / "images" / f"graph_{N_GAMES}_no_labels.png")))